# Problem 2: Predict Daily Return Rankings on Russell 1000
## Notebook 01: Pipeline Verification & Baseline Strategy Evaluation

Mục tiêu của notebook này là kiểm tra tính toàn vẹn của Data Pipeline, kiểm chứng tính chất Dollar-Neutral Long-Short Portfolio và đánh giá hiệu suất mô hình cơ sở (**Baseline 5-Day Reversal**).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Đảm bảo đường dẫn module src được nhận diện
sys.path.append(os.path.abspath(".."))
from src.verify_pipeline import (
    verifyDataIntegrity,
    computeBaselineFiveDayReversal,
    constructDollarNeutralPortfolio,
    evaluateStrategyPerformance,
    printSummaryTable
)

print("Các module đã được import thành công.")

### 1. Đọc và Kiểm định Tính Toàn vẹn Dữ liệu In-Sample
- Số lượng mã tối thiểu: >= 500 mã đạt chuẩn.
- Mốc thời gian: Đóng băng nghiêm ngặt từ 2014-01-01 đến 2023-12-31.

In [ ]:
inSamplePricePath = os.path.join("..", "data", "processed", "in_sample_prices.parquet")
priceMatrix = verifyDataIntegrity(inSamplePricePath)
print(f"Kích thước ma trận giá In-sample: {priceMatrix.shape}")
priceMatrix.iloc[:5, :8]

### 2. Tính toán Tín hiệu Alpha Cơ sở (Baseline 5-Day Reversal)
$$\text{rawAlpha} = \frac{P_{t-5} - P_{t-1}}{5}$$

In [ ]:
rawAlpha = computeBaselineFiveDayReversal(priceMatrix)
print(f"Kích thước ma trận Alpha thô: {rawAlpha.shape}")
rawAlpha.iloc[5:10, :8]

### 3. Phân bổ Vốn Cân bằng Dollar-Neutral (Demean & L1 Normalization)
- **Demean:** $\text{demeanedAlpha}_i = \alpha_i - \text{mean}(\alpha)$
- **L1 Normalization:** $w_i = \frac{\text{demeanedAlpha}_i}{\sum |\text{demeanedAlpha}_j|}$
- Đảm bảo: Tổng vị thế Long (+0.5) và Short (-0.5) triệt tiêu nhau, tổng vốn ròng bằng 0.

In [ ]:
capitalWeights = constructDollarNeutralPortfolio(rawAlpha)

dailyNetHolding = capitalWeights.sum(axis=1).iloc[5:-1]
dailyLongExposure = capitalWeights.clip(lower=0).sum(axis=1).iloc[5:-1].mean()
dailyShortExposure = capitalWeights.clip(upper=0).sum(axis=1).iloc[5:-1].mean()

print(f"Trị tuyệt đối vị thế ròng tối đa: {dailyNetHolding.abs().max():.2e}")
print(f"Tỷ trọng Long trung bình: {dailyLongExposure:.4f}")
print(f"Tỷ trọng Short trung bình: {dailyShortExposure:.4f}")

### 4. Báo cáo Đánh giá Hiệu suất (Performance & Information Metrics)

In [ ]:
metrics, pnlSeries, rankIcSeries = evaluateStrategyPerformance(priceMatrix, capitalWeights, rawAlpha)
printSummaryTable(metrics)